# Week 1 Day 1 — Website summarizer

Goal: Give it a URL, get back an LLM-written summary.

Steps: scrape → prompts → messages → call → display.

In [1]:
from IPython.display import Markdown, display

from llmx import clients, MODELS, fetch_website_contents

## ENV variable checking

Checking what are the env variables needed but is actually missing

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if not api_key:
    print("No api key found, please review the api key")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


## 1. Scrape

`fetch_website_contents` lives in `src/llmx/scraper.py` — read it, don't just import it.

In [27]:
page = fetch_website_contents("https://edwarddonner.com")
print(page[:500])

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.


## 2. Prompts

System prompt = standing behaviour. User prompt = the thing being asked now.

In [2]:
system_prompt = """
You are an assistant that analyzes the contents of a website and provides a short
summary, ignoring text that might be navigation related. Respond in markdown.
"""


def user_prompt_for(page_text):
    return (
        "Here are the contents of a website. Provide a short summary. "
        "If it includes news or announcements, summarize these too.\n\n" + page_text
    )

## 3. Messages + 4. Call

Your turn — write `summarize(url)` yourself. It should:
1. fetch the page
2. build the two-element messages list
3. call `clients["openai"].chat.completions.create(...)`
4. return `response.choices[0].message.content`

In [3]:
def summarize(url):
    fetched_content = fetch_website_contents(url)
    messages = [
        { "role": "system", "content" : system_prompt },
        { "role": "user", "content" : user_prompt_for(fetched_content) }
    ]
    response = clients['gemini'].chat.completions.create(model=MODELS["gemini"], messages=messages)
    return response.choices[0].message.content

display(Markdown(summarize("https://edwarddonner.com")))

### Summary
This is the personal website of Edward Donner, co-founder and CTO of Nebula.io, former CEO of untapt, and a former Managing Director at JPMorgan. He is an AI enthusiast, coder, and best-selling Udemy instructor whose online courses have reached 900,000 students across 194 countries. The site features information about his background, curriculum, AI projects (such as "Outsmart," an arena for LLMs), and resources for his courses.

### Recent Posts & Announcements
* **February 17, 2026:** AI Coder: Vibe Coder to Agentic Engineer – RESOURCES
* **January 4, 2026:** AI Builder with n8n – Create Agents and Voice Agents – RESOURCES
* **September 15, 2025:** AI Engineering MLOps Track – Deploy AI to Production – RESOURCES
* **May 28, 2025:** Which order to take the AI courses?




## Try it on a JavaScript-rendered site

Expect this to come back near-empty — `requests` doesn't run JS. Worth seeing the failure once.

In [ ]:
# display(Markdown(summarize("https://openai.com")))